# Challenge 2: entrenamiento y selección del modelo

En esta etapa debes construir pipelines reproducibles, comparar modelos y seleccionar un campeón sin utilizar el conjunto de test.

**Entradas:** `train.csv` y `data_contract.json`  
**Salidas:** `champion_model.joblib`, `training_metadata.json` y `cv_results.csv`

## Reglas

- El preprocesamiento debe estar dentro de cada `Pipeline`.
- La selección debe hacerse con validación cruzada.
- No cargues `test.csv` en este notebook.

In [1]:
import json
import joblib
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_validate
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

In [9]:
from pathlib import Path

# Buscamos la carpeta raíz del proyecto
PROJECT_DIR = Path.cwd().resolve()
while not (PROJECT_DIR / "data").exists() and PROJECT_DIR != PROJECT_DIR.parent:
    PROJECT_DIR = PROJECT_DIR.parent

RAW_DATA_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_DIR / "data" / "processed"
ARTIFACTS_DIR = PROJECT_DIR / "artifacts"
REPORTS_DIR = PROJECT_DIR / "reports"

for directory in (PROCESSED_DATA_DIR, ARTIFACTS_DIR, REPORTS_DIR):
    directory.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42

## 1. Carga de artefactos

In [10]:
train_df = pd.read_csv(PROCESSED_DATA_DIR / "train.csv")
contract = json.loads((ARTIFACTS_DIR / "data_contract.json").read_text(encoding="utf-8"))

X_train = train_df[contract["features"]]
y_train = train_df[contract["target"]]

print(X_train.shape, y_train.shape)

(142, 13) (142,)


## 2. Pipeline de procesamiento

In [11]:
feature_columns = contract["features"]

tree_preprocessor = ColumnTransformer(
    transformers=[
        ("num", SimpleImputer(strategy="median"), feature_columns)
    ]
)

scaled_preprocessor = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]), feature_columns)
    ]
)    

**Pregunta:** ¿por qué el `StandardScaler` debe estar dentro del pipeline y no ajustarse antes de la validación cruzada?

> Escribe aquí tu respuesta.

Para evitar la fuga de información.

## 3. Modelos candidatos

In [12]:
models = {
    "DecisionTree": Pipeline([
        ("preprocessor", tree_preprocessor),
        ("classifier", DecisionTreeClassifier(random_state=RANDOM_STATE))
    ]),
    "LogisticRegression": Pipeline([
        ("preprocessor", scaled_preprocessor),
        ("classifier", LogisticRegression(random_state=RANDOM_STATE, max_iter=1000))
    ]),
    "KNN": Pipeline([
        ("preprocessor", scaled_preprocessor),
        ("classifier", KNeighborsClassifier())
    ])
}

## 4. Comparación inicial con validación cruzada

In [13]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
initial_rows = []

for name, model_pipeline in models.items():
    scores = cross_validate(
        model_pipeline,
        X_train,
        y_train,
        cv=cv,
        scoring="accuracy",
        return_train_score=True
    )
    
    mean_train = np.mean(scores["train_score"])
    mean_val = np.mean(scores["test_score"])
    std_val = np.std(scores["test_score"])
    
    initial_rows.append({
        "model": name,
        "mean_train_accuracy": mean_train,
        "mean_val_accuracy": mean_val,
        "std_val_accuracy": std_val
    })

initial_comparison = pd.DataFrame(initial_rows)
print("=== COMPARACIÓN INICIAL DE MODELOS ===")
print(initial_comparison)

=== COMPARACIÓN INICIAL DE MODELOS ===
                model  mean_train_accuracy  mean_val_accuracy  \
0        DecisionTree               1.0000           0.902217   
1  LogisticRegression               1.0000           0.979064   
2                 KNN               0.9771           0.950739   

   std_val_accuracy  
0          0.058617  
1          0.027713  
2          0.035274  


## 5. Optimización de hiperparámetros

In [17]:
param_grids = {
    "DecisionTree": {
        "classifier__max_depth": [3, 5, 7, None],
        "classifier__min_samples_split": [2, 5, 10]
    },
    "LogisticRegression": {
        "classifier__C": [0.01, 0.1, 1.0, 10.0],
        "classifier__solver": ["lbfgs", "saga"]
    },
    "KNN": {
        "classifier__n_neighbors": [3, 5, 7, 9],
        "classifier__weights": ["uniform", "distance"]
    }
}

searches = {}
tuned_rows = []

for name, model_pipeline in models.items():
    grid_search = GridSearchCV(
        estimator=model_pipeline,
        param_grid=param_grids[name],
        cv=cv,
        scoring="accuracy",
        return_train_score=True,
        n_jobs=-1
    )
    grid_search.fit(X_train, y_train)
    searches[name] = grid_search
    
    best_idx = grid_search.best_index_
    best_train_acc = grid_search.cv_results_["mean_train_score"][best_idx]
    best_val_acc = grid_search.best_score_
    std_val_acc = grid_search.cv_results_["std_test_score"][best_idx]
    
    tuned_rows.append({
        "model": name,
        "best_params": str(grid_search.best_params_),
        "mean_train_accuracy": best_train_acc,
        "best_val_accuracy": best_val_acc,
        "std_val_accuracy": std_val_acc
    })

tuned_comparison = pd.DataFrame(tuned_rows)
print("\n=== RESULTADOS DE OPTIMIZACION (GridSearchCV) ===")
print(tuned_comparison)


=== RESULTADOS DE OPTIMIZACION (GridSearchCV) ===
                model                                        best_params  \
0        DecisionTree  {'classifier__max_depth': 5, 'classifier__min_...   
1  LogisticRegression  {'classifier__C': 1.0, 'classifier__solver': '...   
2                 KNN  {'classifier__n_neighbors': 7, 'classifier__we...   

   mean_train_accuracy  best_val_accuracy  std_val_accuracy  
0             1.000000           0.902217          0.058617  
1             1.000000           0.979064          0.027713  
2             0.964773           0.957882          0.040520  


## 6. Selección y persistencia del campeón

In [15]:
# 1. Aqui lo que se hace es encontrar el modelo con mayor best_score_ en Validación Cruzada
best_row = tuned_comparison.loc[tuned_comparison["best_val_accuracy"].idxmax()]
champion_name = best_row["model"]
champion_search = searches[champion_name]

# Aquí e obtiene el mejor estimador ajustado sobre todo X_train, y_train
champion = champion_search.best_estimator_

print(f"Modelo Campeón Seleccionado: {champion_name}")
print(f"Mejor Accuracy en CV: {champion_search.best_score_:.4f}")

# 2. Guardar artefactos
# A. champion_model.joblib
joblib.dump(champion, ARTIFACTS_DIR / "champion_model.joblib")

# B. training_metadata.json
metadata = {
    "champion_name": champion_name,
    "best_params": champion_search.best_params_,
    "best_cv_accuracy": float(champion_search.best_score_),
    "features_used": feature_columns,
    "random_state": RANDOM_STATE
}
(ARTIFACTS_DIR / "training_metadata.json").write_text(
    json.dumps(metadata, indent=4), encoding="utf-8"
)

# C. cv_results.csv
cv_results_df = pd.DataFrame(champion_search.cv_results_)
cv_results_df.to_csv(REPORTS_DIR / "cv_results.csv", index=False)

print("Artefactos del campeón exportados con éxito.")

Modelo Campeón Seleccionado: LogisticRegression
Mejor Accuracy en CV: 0.9791
Artefactos del campeón exportados con éxito.


In [16]:
assert (ARTIFACTS_DIR / "champion_model.joblib").exists()
assert (ARTIFACTS_DIR / "training_metadata.json").exists()
assert (REPORTS_DIR / "cv_results.csv").exists()
print("Challenge 2 completado.")

Challenge 2 completado.


**Justificación de la selección:**

> Explica la decisión considerando desempeño, variabilidad, sobreajuste, complejidad e interpretabilidad.

### Solución: 
Ganó Logistic Reegression, que fue el modelo con mayor exactitud en validación cruzada. El pipeline garantiza un preprocesamiento adecuado sin fuga de datos. La combinación de regularización e hiperparámetros oprtimizados permitió alcanzar un alto desempeño con una baja variabilidad entre los pliegues, lo que controló el sobreajuste al mantener una brecha pequeña entre la exactitud de entrenamiento y la de validación.

## Checklist

- [ ] Todo el procesamiento está dentro de pipelines.
- [ ] Comparé tres familias de modelos.
- [ ] Optimicé hiperparámetros con CV.
- [ ] Elegí el campeón sin usar test.
- [ ] Guardé el pipeline completo y sus metadatos.